# 第14章: 畳み込みニューラルネットワークを最新環境で検証する

この Notebook は、原本 `machine-learning-book/ch14/` の内容を、`uv` 管理下の最新依存関係で継続検証しやすい形へ再構成したものです。原本の前半にある畳み込み・プーリング・CNN 構築の要点は維持しつつ、後半の `MNIST` / `CelebA` / `torchvision` 依存部分は、CI で安定に実行できるローカル同梱データセットへ置き換えています。

## この Notebook で扱う内容

- 離散畳み込みを 1 次元と 2 次元で手実装し、PyTorch の演算結果と照合する。
- パディング、ストライド、プーリング、複数チャネル入力の挙動を小さな例で確認する。
- 損失関数として `BCEWithLogitsLoss` と `CrossEntropyLoss` の違いを整理する。
- 原本の `MNIST` CNN は、同じ手書き数字画像分類という学習目的を保つために `sklearn.datasets.load_digits` ベースの小規模 CNN へ置き換える。
- 原本の CelebA smile 分類は外部データ取得と `torchvision` に依存するため、画像拡張を伴う二値 CNN 分類タスクへ置き換える。

In [ ]:
from importlib.metadata import version
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image, display
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, TensorDataset

PROJECT_ROOT = next(
    (candidate.resolve() for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / 'machine-learning-book').exists()),
    None,
)
assert PROJECT_ROOT is not None, 'machine-learning-book/ を含むプロジェクトルートが見つかりません。'

CH14_DIR = PROJECT_ROOT / 'machine-learning-book' / 'ch14'
FIG_DIR = CH14_DIR / 'figures'
assert FIG_DIR.exists(), f'図版ディレクトリが見つかりません: {FIG_DIR}'

PACKAGE_NAMES = ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'torch', 'nbformat']
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=['パッケージ', 'バージョン'],
)

SEED = 1
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Matplotlib backend: {matplotlib.get_backend()}')
print(f'Using device: {device}')
package_versions

## 原本図版の参照

移行版 Notebook は `src/ch14/` に配置しますが、原本の図版は読み取り専用の `machine-learning-book/ch14/figures/` をそのまま利用します。これにより、原本を編集せずに章の説明文脈を維持できます。

In [ ]:
display(Image(filename=str(FIG_DIR / '14_01.png'), width=700))
display(Image(filename=str(FIG_DIR / '14_10.png'), width=700))

## 離散畳み込みを 1 次元と 2 次元で確認する

原本では `NumPy` と `SciPy` を使って畳み込みの仕組みを確認していました。ここでは依存を増やさず、手実装した畳み込みを `torch.nn.functional.conv1d/conv2d` と照合します。PyTorch の `conv` は相関演算なので、比較時にはカーネルを反転してから渡します。

In [ ]:
def conv1d_manual(x, w, p=0, s=1):
    w_rot = np.asarray(w)[::-1]
    x_padded = np.asarray(x, dtype=np.float32)
    if p > 0:
        x_padded = np.pad(x_padded, pad_width=p)
    out = []
    for start in range(0, x_padded.shape[0] - w_rot.shape[0] + 1, s):
        out.append(np.sum(x_padded[start:start + w_rot.shape[0]] * w_rot))
    return np.asarray(out)


def conv2d_manual(x, w, p=(0, 0), s=(1, 1)):
    w_rot = np.asarray(w)[::-1, ::-1]
    x_arr = np.asarray(x, dtype=np.float32)
    x_padded = np.pad(x_arr, ((p[0], p[0]), (p[1], p[1])), mode='constant')
    out = []
    for i in range(0, x_padded.shape[0] - w_rot.shape[0] + 1, s[0]):
        row = []
        for j in range(0, x_padded.shape[1] - w_rot.shape[1] + 1, s[1]):
            row.append(np.sum(x_padded[i:i + w_rot.shape[0], j:j + w_rot.shape[1]] * w_rot))
        out.append(row)
    return np.asarray(out)

x_1d = np.array([1, 3, 2, 4, 5, 6, 1, 3], dtype=np.float32)
w_1d = np.array([1, 0, 3, 1, 2], dtype=np.float32)
manual_1d = conv1d_manual(x_1d, w_1d, p=2, s=1)
torch_1d = F.conv1d(
    torch.tensor(x_1d).view(1, 1, -1),
    torch.tensor(w_1d[::-1].copy()).view(1, 1, -1),
    padding=2,
).view(-1).numpy()

x_2d = np.array([[1, 3, 2, 4], [5, 6, 1, 3], [1, 2, 0, 2], [3, 4, 3, 2]], dtype=np.float32)
w_2d = np.array([[1, 0, 3], [1, 2, 1], [0, 1, 1]], dtype=np.float32)
manual_2d = conv2d_manual(x_2d, w_2d, p=(1, 1), s=(1, 1))
torch_2d = F.conv2d(
    torch.tensor(x_2d).view(1, 1, 4, 4),
    torch.tensor(w_2d[::-1, ::-1].copy()).view(1, 1, 3, 3),
    padding=1,
).squeeze().numpy()

print('conv1d manual:', manual_1d)
print('conv1d torch :', torch_1d)
print('conv2d manual:', manual_2d, sep='\n')
print('conv2d torch :', torch_2d, sep='\n')
assert np.allclose(manual_1d, torch_1d)
assert np.allclose(manual_2d, torch_2d)

kernel_size = 3
padding = 1
stride = 2
input_size = 8
output_size = (input_size + 2 * padding - kernel_size) // stride + 1
print(f'1D output size formula result: {output_size}')

## プーリングと複数チャネル入力

畳み込み層の後段では、特徴量マップの縮約にプーリングが使われます。また、実画像では RGB のような複数チャネル入力を扱います。ここでは小さなテンソルと付属画像を使って挙動を確認します。

In [ ]:
feature_map = torch.tensor(
    [[[[1.0, 2.0, 0.0, 1.0],
       [3.0, 1.0, 2.0, 2.0],
       [0.0, 1.0, 3.0, 1.0],
       [2.0, 2.0, 1.0, 0.0]]]]
)
max_pooled = F.max_pool2d(feature_map, kernel_size=2)
avg_pooled = F.avg_pool2d(feature_map, kernel_size=2)
print('max pooled:', max_pooled.squeeze(), sep='\n')
print('avg pooled:', avg_pooled.squeeze(), sep='\n')

img = plt.imread(CH14_DIR / 'example-image.png')
img = img[..., :3].astype(np.float32)
img_tensor = torch.tensor(img.transpose(2, 0, 1)).unsqueeze(0)

edge_kernel = torch.tensor([
    [[[1.0, 0.0, -1.0], [1.0, 0.0, -1.0], [1.0, 0.0, -1.0]],
     [[1.0, 0.0, -1.0], [1.0, 0.0, -1.0], [1.0, 0.0, -1.0]],
     [[1.0, 0.0, -1.0], [1.0, 0.0, -1.0], [1.0, 0.0, -1.0]]]
])
edge_response = F.conv2d(img_tensor, edge_kernel, padding=1).squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img)
axes[0].set_title('Example RGB image')
axes[0].axis('off')
axes[1].imshow(edge_response, cmap='gray')
axes[1].set_title('Shared 3-channel filter response')
axes[1].axis('off')
plt.tight_layout()
plt.show()
plt.close(fig)

## 分類損失の確認

原本と同様に、二値分類では `BCEWithLogitsLoss`、多クラス分類では `CrossEntropyLoss` を使うのが現在の PyTorch でも標準的です。確率に変換した後で損失を入れるより、logits を直接渡す方が数値的に安定です。

In [ ]:
binary_logits = torch.tensor([0.8])
binary_target = torch.tensor([1.0])
print('BCEWithLogitsLoss:', nn.BCEWithLogitsLoss()(binary_logits, binary_target).item())
print('BCELoss on sigmoid  :', nn.BCELoss()(torch.sigmoid(binary_logits), binary_target).item())

multiclass_logits = torch.tensor([[1.5, 0.8, 2.1]])
multiclass_target = torch.tensor([2])
print('CrossEntropyLoss   :', nn.CrossEntropyLoss()(multiclass_logits, multiclass_target).item())
print('NLLLoss on logprob :', nn.NLLLoss()(torch.log_softmax(multiclass_logits, dim=1), multiclass_target).item())

## 多クラス CNN: digits データセットで手書き数字を分類する

原本では `torchvision.datasets.MNIST` を用いていましたが、最新環境の最小依存と CI 安定性を優先して `sklearn.datasets.load_digits` に置き換えます。入力画像は 8×8 と小さいものの、畳み込み層・プーリング層・全結合層を積み上げて分類する流れは同じです。

In [ ]:
digits = load_digits()
images = (digits.images / 16.0).astype(np.float32)
labels = digits.target.astype(np.int64)

x_train, x_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=SEED, stratify=labels
)
x_valid, x_test, y_valid, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

x_train_tensor = torch.tensor(x_train).unsqueeze(1)
x_valid_tensor = torch.tensor(x_valid).unsqueeze(1)
x_test_tensor = torch.tensor(x_test).unsqueeze(1)
y_train_tensor = torch.tensor(y_train)
y_valid_tensor = torch.tensor(y_valid)
y_test_tensor = torch.tensor(y_test)

train_dl = DataLoader(
    TensorDataset(x_train_tensor, y_train_tensor),
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
valid_dl = DataLoader(TensorDataset(x_valid_tensor, y_valid_tensor), batch_size=128, shuffle=False)

def make_multiclass_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),
        nn.Flatten(),
        nn.Linear(32 * 2 * 2, 64),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(64, 10),
    )

multiclass_model = make_multiclass_cnn().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(multiclass_model.parameters(), lr=0.01)

history = {'train_loss': [], 'valid_loss': [], 'train_acc': [], 'valid_acc': []}
for epoch in range(12):
    multiclass_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    for x_batch, y_batch in train_dl:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        logits = multiclass_model(x_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(x_batch)
        train_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        train_total += len(x_batch)

    multiclass_model.eval()
    valid_loss = 0.0
    valid_correct = 0
    valid_total = 0
    with torch.no_grad():
        for x_batch, y_batch in valid_dl:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            logits = multiclass_model(x_batch)
            loss = loss_fn(logits, y_batch)
            valid_loss += loss.item() * len(x_batch)
            valid_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            valid_total += len(x_batch)

    history['train_loss'].append(train_loss / train_total)
    history['valid_loss'].append(valid_loss / valid_total)
    history['train_acc'].append(train_correct / train_total)
    history['valid_acc'].append(valid_correct / valid_total)

multiclass_model.eval()
with torch.no_grad():
    test_logits = multiclass_model(x_test_tensor.to(device)).cpu()
test_preds = test_logits.argmax(dim=1).numpy()
test_accuracy = accuracy_score(y_test, test_preds)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['valid_loss'], label='valid')
axes[0].set_title('Digits CNN loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['valid_acc'], label='valid')
axes[1].set_title('Digits CNN accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.0, 1.02)
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()
plt.close(fig)

sample_strip = np.hstack([img for img in x_test[:10]])
fig, ax = plt.subplots(figsize=(12, 2.5))
ax.imshow(sample_strip, cmap='gray_r')
ax.set_title('First 10 test images')
ax.axis('off')
plt.show()
plt.close(fig)

pd.DataFrame({
    '項目': ['Validation accuracy (last epoch)', 'Test accuracy'],
    '値': [round(history['valid_acc'][-1], 4), round(test_accuracy, 4)],
})

## 画像拡張つき二値 CNN 分類

原本後半では CelebA を用いた smile 分類を扱っていました。しかし CelebA は外部データ取得と `torchvision` に依存し、CI で継続検証するには不安定です。ここでは代替として、digits 画像に対して「偶数か奇数か」を判定する二値 CNN を用い、原本の二値分類・データ拡張・`BCEWithLogitsLoss` の流れを残します。

In [ ]:
binary_labels = (labels % 2 == 1).astype(np.float32)
xb_train, xb_temp, yb_train, yb_temp = train_test_split(
    images, binary_labels, test_size=0.3, random_state=SEED, stratify=binary_labels
)
xb_valid, xb_test, yb_valid, yb_test = train_test_split(
    xb_temp, yb_temp, test_size=0.5, random_state=SEED, stratify=yb_temp
)

class AugmentedDigitsDataset(Dataset):
    def __init__(self, images, labels, augment=False):
        self.images = torch.tensor(images, dtype=torch.float32).unsqueeze(1)
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.images[idx].clone()
        y = self.labels[idx]
        if self.augment:
            shift_y = int(torch.randint(-1, 2, (1,)).item())
            shift_x = int(torch.randint(-1, 2, (1,)).item())
            x = torch.roll(x, shifts=(shift_y, shift_x), dims=(1, 2))
            x = torch.clamp(x + 0.05 * torch.randn_like(x), 0.0, 1.0)
        return x, y

train_binary_dl = DataLoader(
    AugmentedDigitsDataset(xb_train, yb_train, augment=True),
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
valid_binary_ds = AugmentedDigitsDataset(xb_valid, yb_valid, augment=False)
test_binary_ds = AugmentedDigitsDataset(xb_test, yb_test, augment=False)
valid_binary_dl = DataLoader(valid_binary_ds, batch_size=128, shuffle=False)
test_binary_dl = DataLoader(test_binary_ds, batch_size=128, shuffle=False)

preview_ds = AugmentedDigitsDataset(xb_train[:4], yb_train[:4], augment=True)
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for col in range(4):
    axes[0, col].imshow(xb_train[col], cmap='gray_r')
    axes[0, col].axis('off')
    aug_img, _ = preview_ds[col]
    axes[1, col].imshow(aug_img.squeeze(0), cmap='gray_r')
    axes[1, col].axis('off')
axes[0, 0].set_title('Original')
axes[1, 0].set_title('Augmented')
plt.tight_layout()
plt.show()
plt.close(fig)

binary_model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Dropout(p=0.2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(64, 1),
).to(device)

binary_loss_fn = nn.BCEWithLogitsLoss()
binary_optimizer = torch.optim.Adam(binary_model.parameters(), lr=0.01)

binary_history = {'train_loss': [], 'valid_loss': [], 'train_acc': [], 'valid_acc': []}
for epoch in range(10):
    binary_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    for x_batch, y_batch in train_binary_dl:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        binary_optimizer.zero_grad()
        logits = binary_model(x_batch).squeeze(1)
        loss = binary_loss_fn(logits, y_batch)
        loss.backward()
        binary_optimizer.step()
        train_loss += loss.item() * len(x_batch)
        train_correct += ((torch.sigmoid(logits) >= 0.5) == y_batch.bool()).sum().item()
        train_total += len(x_batch)

    binary_model.eval()
    valid_loss = 0.0
    valid_correct = 0
    valid_total = 0
    with torch.no_grad():
        for x_batch, y_batch in valid_binary_dl:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            logits = binary_model(x_batch).squeeze(1)
            loss = binary_loss_fn(logits, y_batch)
            valid_loss += loss.item() * len(x_batch)
            valid_correct += ((torch.sigmoid(logits) >= 0.5) == y_batch.bool()).sum().item()
            valid_total += len(x_batch)

    binary_history['train_loss'].append(train_loss / train_total)
    binary_history['valid_loss'].append(valid_loss / valid_total)
    binary_history['train_acc'].append(train_correct / train_total)
    binary_history['valid_acc'].append(valid_correct / valid_total)

binary_model.eval()
all_test_logits = []
all_test_targets = []
with torch.no_grad():
    for x_batch, y_batch in test_binary_dl:
        logits = binary_model(x_batch.to(device)).squeeze(1).cpu()
        all_test_logits.append(logits)
        all_test_targets.append(y_batch)

test_logits_binary = torch.cat(all_test_logits)
test_targets_binary = torch.cat(all_test_targets)
test_probs_binary = torch.sigmoid(test_logits_binary)
test_preds_binary = (test_probs_binary >= 0.5).to(torch.int64).numpy()
test_acc_binary = accuracy_score(test_targets_binary.numpy().astype(np.int64), test_preds_binary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(binary_history['train_loss'], label='train')
axes[0].plot(binary_history['valid_loss'], label='valid')
axes[0].set_title('Binary CNN loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(binary_history['train_acc'], label='train')
axes[1].plot(binary_history['valid_acc'], label='valid')
axes[1].set_title('Binary CNN accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.0, 1.02)
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()
plt.close(fig)

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, prob, target in zip(axes.flat, xb_test[:10], test_probs_binary[:10], yb_test[:10]):
    ax.imshow(image, cmap='gray_r')
    label = 'odd' if int(target) == 1 else 'even'
    ax.set_title(f'{label}, p={prob.item():.2f}')
    ax.axis('off')
plt.tight_layout()
plt.show()
plt.close(fig)

pd.DataFrame({
    '項目': ['Validation accuracy (last epoch)', 'Test accuracy'],
    '値': [round(binary_history['valid_acc'][-1], 4), round(test_acc_binary, 4)],
})

## まとめ

この移行版では、原本 `ch14` の主題である畳み込み、プーリング、複数チャネル、分類損失、CNN 学習を、現行の `torch` と `scikit-learn` だけで継続検証できる形に再構成しました。原本の `machine-learning-book/ch14/` は参照のみで、サブモジュール内のファイルは変更していません。